In [4]:
import pandas as pd
import ast
import logging

# İşlem takibi için loglama yapılandırılması
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def integrate_temporal_data(cleaned_path, raw_path, output_path):
    """
    Temizlenmiş veri setini ham veri setindeki zaman damgalarıyla birleştirir.
    Zaman serisi analizi (Prophet) için veri setini hazır hale getirir.
    """
    try:
        logging.info("Veri kaynakları yükleniyor...")
        df_cleaned = pd.read_csv(cleaned_path)
        df_raw = pd.read_csv(raw_path)

        # ID kolonlarını standartlaştırma (Eşleşme doğruluğu için)
        df_cleaned['id'] = df_cleaned['id'].astype(str).str.lower().str.strip()
        df_raw['id'] = df_raw['id'].astype(str).str.lower().str.strip()

        # Metadata içerisinden zaman damgasını ayıklama
        def extract_timestamp(post_content):
            try:
                # String formatındaki sözlük yapısını güvenli değerlendirme
                return ast.literal_eval(post_content).get('created_at')
            except (ValueError, SyntaxError, AttributeError):
                return None

        logging.info("Zaman damgaları metaveriden ayıklanıyor...")
        df_raw['timestamp'] = df_raw['post'].apply(extract_timestamp)

        # Temizlenmiş veri ile zaman damgalarını birleştirme (Inner Join)
        logging.info("Veri kümeleri birleştiriliyor...")
        df_final = pd.merge(df_cleaned, df_raw[['id', 'timestamp']], on='id', how='inner')

        # Zaman formatını standart YYYY-MM-DD biçimine dönüştürme
        df_final['timestamp'] = pd.to_datetime(df_final['timestamp']).dt.date

        # Geçersiz zaman verilerini temizleme ve dışa aktarma
        df_final = df_final.dropna(subset=['timestamp'])
        df_final.to_csv(output_path, index=False)

        logging.info(f"İşlem başarıyla tamamlandı. Dosya: {output_path}")
        logging.info(f"Toplam işlenen satır sayısı: {len(df_final)}")

        return df_final.head()

    except Exception as e:
        logging.error(f"İşlem sırasında bir hata oluştu: {e}")

if __name__ == "__main__":
    # Dosya yolları tanımları
    CLEANED_FILE = 'moltbook_temiz.csv'
    RAW_FILE = 'moltbook_ham.csv'
    OUTPUT_FILE = 'moltbook_temiz_final.csv'

    sample = integrate_temporal_data(CLEANED_FILE, RAW_FILE, OUTPUT_FILE)
    if sample is not None:
        print("\nİşlenen Veri Örneği:")
        print(sample)


İşlenen Veri Örneği:
                                     id topic_label  toxic_level  \
0  8c3baf32-6b12-49e0-9326-a72123b6df08           E            0   
1  3b81b374-6cd6-43ee-82fd-31c9c57eb534           A            0   
2  b5e85b61-61b3-4e5f-9291-c6372d21efd6           B            0   
3  f2b65193-79de-4525-8a19-e095e0314740           D            0   
4  b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1           H            0   

                                                post   timestamp  
0  {'comment_count': 0, 'content': 'Spent the aft...  2026-01-31  
1  {'comment_count': 0, 'content': 'Its 22:55 UTC...  2026-01-31  
2  {'comment_count': 0, 'content': '```typescript...  2026-01-31  
3  {'comment_count': 0, 'content': 'Just helped a...  2026-01-31  
4  {'comment_count': 0, 'content': 'This is a tes...  2026-01-31  
